# Market Silver EDA
EDA riêng cho coverage, chất lượng OHLCV, return và volatility của VN30.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = Path.cwd().parent if Path.cwd().name == 'eda' else Path.cwd()
market = pd.read_parquet(ROOT / 'data_lake/silver/market_daily')
market['session_date'] = pd.to_datetime(market.session_date.astype(str))
print(f'Market rows: {len(market):,}')

In [ ]:
quality = pd.Series({
    'rows': len(market),
    'tickers': market.ticker.nunique(),
    'sessions': market.session_date.nunique(),
    'first_session': market.session_date.min(),
    'last_session': market.session_date.max(),
    'duplicate_keys': market.duplicated(['ticker','session_date','interval','source']).sum(),
    'invalid_rows': (~market.is_valid).sum(),
    'weekend_rows': market.session_date.dt.dayofweek.ge(5).sum(),
    'missing_daily_return': market.daily_return.isna().sum(),
    'abs_return_gt_10pct': market.daily_return.abs().gt(.10).sum(),
}, name='value').to_frame()
display(quality)

In [ ]:
coverage = market.groupby('ticker').agg(
    sessions=('session_date','nunique'),
    first_session=('session_date','min'),
    last_session=('session_date','max'),
    missing_return=('daily_return', lambda s: s.isna().sum()),
).sort_values('sessions')
display(coverage)
coverage.sessions.plot.bar(figsize=(13,4), title='Số phiên theo ticker')
plt.ylabel('Phiên'); plt.tight_layout(); plt.show()

In [ ]:
returns = market.daily_return.dropna()
return_summary = returns.describe(percentiles=[.01,.05,.5,.95,.99]).to_frame('value')
return_summary['unit'] = '%'
return_summary.loc['count', 'unit'] = 'rows'
return_summary.loc[return_summary.index != 'count', 'value'] *= 100
display(return_summary.rename(columns={'value':'daily_return'}))

In [ ]:
ticker_risk = market.groupby('ticker').agg(
    mean_return=('daily_return','mean'),
    volatility=('daily_return','std'),
    median_volume=('volume','median'),
).sort_values('volatility', ascending=False)
display(ticker_risk.assign(mean_return=lambda x:100*x.mean_return, volatility=lambda x:100*x.volatility))
fig, axes = plt.subplots(1,2,figsize=(14,5))
(100*returns).clip(-15,15).plot.hist(bins=60, ax=axes[0], title='Daily return (%)')
(100*ticker_risk.volatility).sort_values().plot.barh(ax=axes[1], title='Volatility theo ticker (%)')
plt.tight_layout(); plt.show()

In [ ]:
return_matrix = market.pivot(index='session_date', columns='ticker', values='daily_return')
corr = return_matrix.corr(min_periods=100)
fig, ax = plt.subplots(figsize=(10,8))
image = ax.imshow(corr, vmin=-1, vmax=1, cmap='RdBu_r')
ax.set_xticks(range(len(corr)), corr.columns, rotation=90)
ax.set_yticks(range(len(corr)), corr.index)
ax.set_title('Tương quan daily return')
fig.colorbar(image, ax=ax); plt.tight_layout(); plt.show()